<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-08-tool-use-and-mcp/lesson-8.1-function-calling-for-mcp/notebooks/exercises-8_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8.1: From Text to Action: Function Calling

*Module 8 · MCP, Tool Orchestration & Function Calling LIVE CLASS*

> Module 5 introduced function calling. Module 8 masters it. This lesson puts all three providers side-by-side — Gemini, OpenAI, and Claude — comparing their schemas, execution loops, parallel calls, and error handling. Then we build a provider-agnostic ToolExecutor that works across all three, setting the foundation for MCP in lesson 8.2.

`3 Providers` · `Schema Design` · `Parallel Calls` · `Error Handling` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-8.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Side-by-Side — Three Providers, Same Task — `01_three_providers.py`
2. Step 2 — Execution Loops — The Full Cycle for Each Provider — `02_execution_loops.py`
3. Step 3 — Universal ToolExecutor — Provider-Agnostic — `03_universal_executor.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q anthropic>=0.40.0 google-generativeai openai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export ANTHROPIC_API_KEY=sk-...
#   export GOOGLE_API_KEY=sk-...
#   export OPENAI_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("ANTHROPIC_API_KEY", "YOUR_ANTHROPIC_API_KEY_HERE")
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")
os.environ.setdefault("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Side-by-Side — Three Providers, Same Task

Same function, three different schemas. See the differences clearly.

**`01_three_providers.py`** · _Block 1 of 3_

FUNCTION CALLING — 3 Providers Side-by-Side


In [ ]:
# FUNCTION CALLING — 3 Providers Side-by-Side
import json

# The SAME function, three schema formats

# ── Gemini: Python function with type hints (auto-schema) ──
def get_weather_gemini(city: str, unit: str = "celsius") -> dict:
    """Get current weather for a city.

    Args:
        city: The city name (e.g. 'Hyderabad')
        unit: Temperature unit, 'celsius' or 'fahrenheit'
    """
    return {"city": city, "temp": 34, "unit": unit, "condition": "Sunny"}

# ── OpenAI: JSON Schema ──
openai_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
                "unit": {"type": "string", "enum": ["celsius","fahrenheit"], "default": "celsius"}
            },
            "required": ["city"]
        }
    }
}

# ── Anthropic: JSON Schema (similar to OpenAI) ──
anthropic_tool = {
    "name": "get_weather",
    "description": "Get current weather for a city",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name"},
            "unit": {"type": "string", "enum": ["celsius","fahrenheit"]}
        },
        "required": ["city"]
    }
}

print("Function Calling Schemas — 3 Providers:\n")
print("  Gemini:    Python function + docstring (auto-generates schema)")
print("  OpenAI:    JSON Schema in 'parameters' key")
print("  Anthropic: JSON Schema in 'input_schema' key\n")
print("  Key differences:")
print("    Gemini:    tools=[python_function] (simplest)")
print("    OpenAI:    tools=[{type:'function', function:{...}}]")
print("    Anthropic: tools=[{name:'...', input_schema:{...}}]")
print("    Response:  Gemini=auto | OpenAI=tool_calls[] | Anthropic=content[type=tool_use]")


### Step 2 · Execution Loops — The Full Cycle for Each Provider

Call LLM → detect tool request → execute → send result back → get final answer.

**`02_execution_loops.py`** · _Block 2 of 3_

EXECUTION LOOPS — Each provider's call-execute-return cycle


In [ ]:
# EXECUTION LOOPS — Each provider's call-execute-return cycle
import json, os

# Shared function
def execute_tool(name, args):
    tools = {
        "get_weather": lambda city, unit="celsius": {"temp":34,"city":city,"cond":"Sunny"},
        "calculate": lambda expr: {"result": eval(expr, {"__builtins__":{}})},
    }
    return json.dumps(tools[name](**args))

# ── Gemini Loop (simplest: automatic) ──
def gemini_loop(query):
    import google.generativeai as genai
    genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
    def get_weather(city: str, unit: str="celsius") -> dict:
        """Get current weather."""
        return {"temp":34,"city":city,"cond":"Sunny"}
    model = genai.GenerativeModel("gemini-2.0-flash", tools=[get_weather])
    chat = model.start_chat(enable_automatic_function_calling=True)
    return chat.send_message(query).text

# ── OpenAI Loop (manual tool_calls handling) ──
def openai_loop(query):
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    msgs = [{"role":"user","content":query}]
    resp = client.chat.completions.create(model="gpt-4o", messages=msgs, tools=[openai_tool])
    msg = resp.choices[0].message
    if msg.tool_calls:
        msgs.append(msg)
        for tc in msg.tool_calls:
            result = execute_tool(tc.function.name, json.loads(tc.function.arguments))
            msgs.append({"role":"tool", "tool_call_id":tc.id, "content":result})
        return client.chat.completions.create(model="gpt-4o",messages=msgs).choices[0].message.content
    return msg.content

# ── Anthropic Loop (content blocks with tool_use) ──
def anthropic_loop(query):
    import anthropic
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    resp = client.messages.create(model="claude-sonnet-4-20250514", max_tokens=500,
        tools=[anthropic_tool], messages=[{"role":"user","content":query}])
    tool_blocks = [b for b in resp.content if b.type=="tool_use"]
    if tool_blocks:
        results = [{"type":"tool_result", "tool_use_id":b.id,
                    "content":execute_tool(b.name, b.input)} for b in tool_blocks]
        final = client.messages.create(model="claude-sonnet-4-20250514", max_tokens=500,
            tools=[anthropic_tool], messages=[{"role":"user","content":query},
            {"role":"assistant","content":resp.content}, {"role":"user","content":results}])
        return final.content[0].text
    return resp.content[0].text

print("Execution Loops:\n")
print("  Gemini:    enable_automatic_function_calling=True (zero boilerplate)")
print("  OpenAI:    check msg.tool_calls → execute → append role='tool' → re-call")
print("  Anthropic: filter content for type='tool_use' → execute → send tool_result")


> **What just happened?** Same weather query, three execution loops. Gemini is simplest: one flag enables auto-calling. OpenAI requires a manual loop: detect tool_calls , execute, append as role="tool" , re-call. Anthropic filters content for type="tool_use" blocks, then sends tool_result . The concept is identical. The plumbing differs. This is why we build a universal executor next.


### Step 3 · Universal ToolExecutor — Provider-Agnostic

One class that handles function calling across all three providers.

**`03_universal_executor.py`** · _Block 3 of 3_

UNIVERSAL TOOL EXECUTOR — Works with any provider


In [ ]:
# UNIVERSAL TOOL EXECUTOR — Works with any provider
import json, time
from dataclasses import dataclass, field

@dataclass
class ToolCall:
    name: str
    args: dict
    id: str = ""

@dataclass
class ToolResult:
    name: str
    output: str
    success: bool = True
    latency_ms: float = 0

class ToolExecutor:
    """Register tools, execute safely, log everything."""
    def __init__(self):
        self.tools = {}
        self.log = []

    def register(self, func, description="", retries=2):
        self.tools[func.__name__] = {"func":func, "desc":description, "retries":retries}
        return func

    def execute(self, call: ToolCall) -> ToolResult:
        tool = self.tools.get(call.name)
        if not tool:
            return ToolResult(call.name, json.dumps({"error":f"Unknown: {call.name}"}), False)
        for attempt in range(tool["retries"]):
            try:
                start = time.time()
                result = tool["func"](**call.args)
                lat = (time.time()-start)*1000
                tr = ToolResult(call.name, json.dumps(result), True, lat)
                self.log.append(tr); return tr
            except Exception as e:
                if attempt == tool["retries"]-1:
                    tr = ToolResult(call.name, json.dumps({"error":str(e)}), False)
                    self.log.append(tr); return tr
                time.sleep(0.5)

    def execute_parallel(self, calls: list[ToolCall]) -> list[ToolResult]:
        """Execute multiple tool calls (sequentially here; use asyncio for true parallel)."""
        return [self.execute(c) for c in calls]

    def get_schemas_openai(self):
        """Export tool schemas in OpenAI format."""
        return [{"type":"function","function":{"name":n,"description":t["desc"]}} for n,t in self.tools.items()]

    def get_functions_gemini(self):
        """Export tool functions for Gemini."""
        return [t["func"] for t in self.tools.values()]

# ── Register tools ──
executor = ToolExecutor()

@executor.register
def get_weather(city: str) -> dict:
    """Get current weather for a city."""
    data = {"Hyderabad":{"temp":34,"cond":"Sunny"}, "Mumbai":{"temp":30,"cond":"Humid"}}
    return data.get(city, {"error":"Unknown city"})

@executor.register
def calculate(expression: str) -> dict:
    """Evaluate a math expression."""
    return {"result": eval(expression, {"__builtins__":{}})}

# ── Execute ──
print("Universal ToolExecutor:\n")
calls = [ToolCall("get_weather", {"city":"Hyderabad"}), ToolCall("calculate", {"expression":"14999*1.18"})]
results = executor.execute_parallel(calls)
for r in results:
    print(f"  {r.name}: {r.output} ({r.latency_ms:.0f}ms, {'OK' if r.success else 'FAIL'})")
print(f"\n  Works with: Gemini (get_functions_gemini), OpenAI (get_schemas_openai), Anthropic")


> **What just happened?** Schema quality directly determines tool calling reliability. Descriptions tell the model WHEN to call a tool. Enums constrain WHAT values it can generate. Required fields ensure the model doesn't skip critical parameters. These details make the difference between 70% and 95% tool calling accuracy.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
